# HH-RLHF: ridge baseline and memory-size ablation

This notebook runs the saved-vector CPU experiment from the organized project. It adds no PPO training or grader calls. See `docs/hh_offline/README.md` for the frozen comparison protocol.


In [ ]:
from pathlib import Path
import subprocess, sys, json
from IPython.display import Markdown, display

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hh_offline.py").is_file()), None)
if PROJECT is None:
    candidate = Path.cwd() / "workshop_project"
    if (candidate / "hh_offline.py").is_file():
        PROJECT = candidate
if PROJECT is None:
    raise RuntimeError("Open this notebook inside the workshop_project checkout.")
PROJECT = PROJECT.resolve()
print(PROJECT)


Install `requirements-hh-offline.txt` in this notebook's Python environment if needed. The saved input files must also be present; a source-only Git clone does not contain the large evaluation data.


In [ ]:
command = [sys.executable, str(PROJECT / "hh_offline.py"), "run"]
preview = subprocess.run(command + ["--dry-run"], cwd=PROJECT, text=True, check=True, capture_output=True)
plan = json.loads(preview.stdout)
print(json.dumps(plan, indent=2))


In [ ]:
# Fits only saved training-memory labels, tunes on validation, then evaluates.
subprocess.run(command, cwd=PROJECT, check=True)


In [ ]:
output = Path(plan["output"])
display(Markdown((output / "report.md").read_text(encoding="utf-8")))


In [ ]:
import pandas as pd
from IPython.display import Image
display(Image(filename=str(output / "memory_ablation.png")))
metrics = pd.read_csv(output / "metrics.csv")
display(metrics[(metrics.cohort == "test") & (metrics.fraction == 1.0)])
